# 🔄 Data Transformation with Spark SQL in Databricks — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![PySpark](https://img.shields.io/badge/Library-PySpark-E25A1C?logo=apachespark&logoColor=white)
![Delta Lake](https://img.shields.io/badge/Storage-Delta%20Lake-00ADD8)
![Dataset](https://img.shields.io/badge/Dataset-Online%20Retail-blueviolet)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Platform:** DataCamp | **Track:** Associate Data Engineer in Databricks (Course 13)
> **Instructor:** Disha Mukherjee

**Scenario I worked through:** a data engineer at **Global Retail Analytics**, an e-commerce consulting firm, building an end-to-end ETL pipeline on the `online_retail` dataset (108,361 transactions) in a **live Azure Databricks workspace** — from loading raw CSV files in a Unity Catalog volume to a scheduled, multi-task production job.

**Objectives**
1. Load, inspect and shape data in Databricks notebooks with PySpark and Spark SQL.
2. Build a cleaning pipeline (explicit schema, nulls, duplicates, invalid records) and optimize joins with broadcast.
3. Rank with window functions, ingest incrementally with Structured Streaming and checkpoints, and persist to Delta Lake.
4. Deploy the pipeline as a Databricks Job and understand when to move to Lakeflow.

Code cells are transcriptions of the notebooks I ran in Databricks (`spark` and `dbutils` are provided there), so they are not meant to be re-run here. Volume paths are shown with `databricks_ws_***` in place of the workspace-specific catalog name. Results appear as screenshots in `./screenshots/`.

| Chapter | Topic |
|---|---|
| 1 | Loading and shaping data |
| 2 | Data cleaning and optimization |
| 3 | Analytics and production pipelines |

## Chapter 1 — Loading and Shaping Data

### 1.1 — Databricks notebooks and data in Unity Catalog volumes

Databricks provides engineering and analytics tools on top of Spark and handles cluster provisioning and scaling. The exercises ran on **serverless compute** — same architecture, no setup.

A Databricks notebook is made of **independent cells** that run SQL or Python and display results inline — unlike a traditional SQL script that runs top to bottom as one unit. Raw files live in **Unity Catalog volumes**, addressed as `/Volumes/<catalog>/<schema>/<volume>/<file>`.

In [ ]:
path = "/Volumes/.../default/transactions.csv"
df = spark.read.csv(path, header=True, inferSchema=True)

df.printSchema()             # column names, types, nullability
df.show(5, truncate=False)   # preview rows
display(df)                  # Databricks interactive table: sort, filter, page, spot nulls and outliers

**Exercise — loading the first dataset.** I found `online_retail.csv` in the `datasets` volume of my workspace catalog, loaded it and checked its schema:

In [ ]:
# Replace with your own path
path = "/Volumes/databricks_ws_***/default/datasets/online_retail.csv"

# Read the CSV file
df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

df.printSchema()

![online_retail columns in Catalog Explorer: InvoiceNo, StockCode, Description (string), Quantity (int), InvoiceDate (timestamp), UnitPrice (double)…](./screenshots/ch1_ex2_catalog_online_retail_schema.png)

*online_retail columns in Catalog Explorer: InvoiceNo, StockCode, Description (string), Quantity (int), InvoiceDate (timestamp), UnitPrice (double)…*

**Result:** `UnitPrice` is a **double**.

**Exercise — exploring driver logs.** After previewing 10 rows (`df.show(10)`), I opened the **driver logs** from the compute menu — the first place to look when a Spark job fails.

![Driver log files: stdout, stderr and log4j output, including rotated files](./screenshots/ch1_ex3_driver_logs.png)

*Driver log files: stdout, stderr and log4j output, including rotated files*

**Interpretation:** `stdout` holds what the code prints, `stderr` holds errors and stack traces, and `log4j` holds Spark's own engine logs. Rotated files (with timestamps, or `.gz`) keep older history.

### 1.2 — Shaping data with PySpark and SQL

`select()` chooses columns and `filter()` chooses rows. Conditions are combined with `&` (and) / `|` (or), each wrapped in parentheses.

In [ ]:
from pyspark.sql import functions as F

df.select("ID", "Customer_ID", "Transaction_Amount", "Category").show(5)

filtered_df = df.filter(F.col("Transaction_Status") == "Completed")

high_value_df = df.filter(
    (F.col("Transaction_Amount") > 20000) &
    (F.col("Transaction_Status") == "Completed")
)

**Python methods or SQL?** Python methods build transformations step by step; SQL is often more readable for aggregations and grouping. A **temporary view** is a name pointing to a DataFrame, and it lasts only for the current Spark session.

In [ ]:
df.createOrReplaceTempView("transactions")

```sql
%sql
SELECT Category, COUNT(*) AS transaction_count
FROM transactions
WHERE Transaction_Status = 'Completed'
GROUP BY Category
ORDER BY transaction_count DESC
```

`df.explain()` (or `EXPLAIN SELECT …` in SQL) shows the physical plan, which helps diagnose performance. Spark errors are also worth reading carefully: selecting a non-existent column returns `[UNRESOLVED_COLUMN.WITH_SUGGESTION]` together with a list of similar existing columns.

**Exercise — using PySpark to shape data.** The French sales team flagged unusual bulk orders, so I selected the relevant columns and kept orders from France with quantity above 20:

In [ ]:
from pyspark.sql import functions as F

# Select specified columns
selected_df = df.select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Country",
    "Quantity",
    "UnitPrice"
)

selected_df.show(5, truncate=False)

![The six selected columns](./screenshots/ch1_ex4_selected_columns.png)

*The six selected columns*

In [ ]:
# Filter for rows where Quantity > 20 AND Country is "France"
filtered_df = selected_df.filter(
    (F.col("Quantity") > 20) &
    (F.col("Country") == "France")
)

# Display the filtered results
display(filtered_df)

![Using the interactive table filter to find invoice 536998](./screenshots/ch1_ex4_invoice_536998_filter.png)

*Using the interactive table filter to find invoice 536998*

**Result:** invoice **536998** ordered **58** units of *MINI JIGSAW QUEEN AND COUNTRY*. The `display()` table filter answered the question without writing another query.

**Exercise — analyzing data with SQL.** To study cancellation patterns by country (cancelled invoices start with `C`), I filtered in Python, registered a view and aggregated in SQL:

In [ ]:
from pyspark.sql import functions as F

# Selecting only cancelled sales
cancelled_sales = df.filter(F.col("InvoiceNo").startswith("C"))

# Creating a temporary view
cancelled_sales.createOrReplaceTempView("sales")

```sql
%sql
SELECT Country, COUNT(*) AS order_count
-- Selecting data from the temporary view
FROM sales
GROUP BY Country
ORDER BY order_count DESC
```

![Lower end of the cancellation ranking: countries with 1–2 cancelled orders](./screenshots/ch1_ex5_cancellations_by_country.png)

*Lower end of the cancellation ranking: countries with 1–2 cancelled orders*

```sql
%sql
-- Checking the execution plan
EXPLAIN
SELECT Country, COUNT(*) AS order_count
FROM sales
GROUP BY Country
ORDER BY order_count DESC
```

![Photon plan: the Python filter is pushed down into the scan as StartsWith(InvoiceNo, C)](./screenshots/ch1_ex5_explain_datafilters.png)

*Photon plan: the Python filter is pushed down into the scan as StartsWith(InvoiceNo, C)*

**Interpretation:** the `DataFilters` section shows **`StartsWith(InvoiceNo, C)`** applied directly in the `PhotonScan`, and `ReadSchema` lists only `InvoiceNo` and `Country`. Even though the filter was written in Python and the aggregation in SQL, Spark combined them into one optimized plan that reads only two columns and filters at the source. The `Photon` prefix shows the query ran on Databricks' native vectorized engine.

**Exercise — understanding temporary views.** A colleague opened a new notebook session the next day and queried a view created with `createOrReplaceTempView("retail_filtered")`:

```sql
%sql
SELECT * FROM retail_filtered
LIMIT 10
-- [TABLE_OR_VIEW_NOT_FOUND] The table or view `retail_filtered` cannot be found.
```

**Explanation:** the view was **session-scoped** and disappeared when the original Spark session ended. Anything that must survive sessions should be saved as a table (see Delta tables in Chapter 3).

## Chapter 2 — Data Cleaning and Optimization

### 2.1 — Why explicit schemas matter

Real data has three recurring problems: **missing values** in key columns, **duplicate rows** from repeated ingestion, and **logically invalid records**. Cleaning them reliably starts with types: Spark's inference can misread columns (e.g. numbers as strings), so a `StructType` schema fixes the types on every run.

**Exercise — explaining a row-count discrepancy.** Two engineers ran the same cleaning steps on `customer_orders.csv`:

In [ ]:
# Colleague's approach — inferred schema
df = spark.read.csv(path, header=True, inferSchema=True)
df_clean = df.na.drop(subset=["CustomerID"])\
             .na.fill({"OrderTotal": 0.0}).dropDuplicates()
df_clean.count()   # -> 45,230 rows

# My approach — explicit schema
schema = StructType([
    StructField("OrderID",    StringType(),    True),
    StructField("CustomerID", DoubleType(),    True),
    StructField("OrderDate",  TimestampType(), True),
    StructField("OrderTotal", DoubleType(),    True),
])
df = spark.read.csv(path, header=True, schema=schema)
df_clean = df.na.drop(subset=["CustomerID"])\
             .na.fill({"OrderTotal": 0.0}).dropDuplicates()
df_clean.count()   # -> 42,890 rows

**Explanation:** without a schema, Spark guessed the wrong column types, so the cleaning steps couldn't correctly identify invalid values. For example, `na.fill({"OrderTotal": 0.0})` only applies to numeric columns, and malformed values in a column inferred as string are not treated as nulls. With explicit types, invalid values become proper nulls and are handled — hence fewer, but correct, rows.

### 2.2 — The cleaning toolkit

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql import functions as F

# Count nulls in every column in one pass
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])

df_no_nulls = df.na.drop(subset=["Customer_ID"])                       # key column must exist
df_filled = df_no_nulls.na.fill({"Transaction_Amount": 0.0,            # safe defaults elsewhere
                                 "Category": "Unknown", "Location": "Unknown"})

duplicates = df_filled.count() - df_filled.distinct().count()           # exact duplicates
df_deduped = df_filled.dropDuplicates(subset=["Customer_ID", "Date", "Transaction_Amount"])  # business key

df_valid = df_deduped.filter(F.col("Transaction_Amount") > 0)          # logically invalid records
df_valid = df_valid.filter(F.col("Transaction_Status") == "Completed")

df_enriched = df_valid.withColumn("Revenue_Band",                      # derived column
    F.when(F.col("Transaction_Amount") >= 50000, "High")
     .when(F.col("Transaction_Amount") >= 10000, "Medium")
     .otherwise("Low"))

# Final quality check: rows, duplicates and null rate of the key column
total_rows = df_enriched.count()
distinct_rows = df_enriched.distinct().count()
null_rate = df_enriched.filter(F.col("Customer_ID").isNull()).count() / total_rows * 100
print(f"Total rows: {total_rows}")
print(f"Duplicates: {total_rows - distinct_rows}")
print(f"Null rate: {null_rate:.2f}%")

**Exercise — cleaning the online retail dataset.** Missing customer IDs, duplicated rows and cancelled orders were skewing weekly reports.

**Step 1 — imports and path**

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql import functions as F

# Replace with your own path
path = "/Volumes/databricks_ws_***/default/course_datasets/online_retail.csv"

**Step 2 — define the schema and load**

In [ ]:
# Define schema with all 8 columns
schema = StructType([
    StructField("InvoiceNo",   StringType(),    True),
    StructField("StockCode",   StringType(),    True),
    StructField("Description", StringType(),    True),
    StructField("Quantity",    IntegerType(),   True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice",   DoubleType(),    True),
    StructField("CustomerID",  DoubleType(),    True),
    StructField("Country",     StringType(),    True),
])

# Load CSV with explicit schema
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(path)
)

df.printSchema()

![Schema exactly as defined: Quantity integer, InvoiceDate timestamp, UnitPrice and CustomerID double](./screenshots/ch2_ex2_explicit_schema.png)

*Schema exactly as defined: Quantity integer, InvoiceDate timestamp, UnitPrice and CustomerID double*

**Step 3 — diagnose nulls, duplicates, negative quantities and cancellations**

In [ ]:
# Count nulls per column
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
)
null_counts.show()

# Count duplicate rows
duplicates = df.count() - df.distinct().count()
print(f"Duplicate rows        : {duplicates:,}")

# Count rows with zero or negative Quantity
invalid_qty = df.filter(F.col("Quantity") <= 0).count()
print(f"Invalid quantity rows : {invalid_qty:,}")

# Count cancellation rows (InvoiceNo starts with 'C')
cancellations = df.filter(F.col("InvoiceNo").startswith("C")).count()
print(f"Cancellation rows     : {cancellations:,}")

![Data quality diagnosis before cleaning](./screenshots/ch2_ex2_quality_checks.png)

*Data quality diagnosis before cleaning*

**Interpretation:** the biggest problem is **26,894 rows without `CustomerID`** (about 25% of the data); `Description` has 392 nulls; there are 100 exact duplicates, 595 rows with zero or negative quantity and 369 cancellations.

**Step 4 — clean**

In [ ]:
rows_before = df.count()

# Remove rows where CustomerID is null
df_clean = df.na.drop(subset=["CustomerID"])

# Fill remaining nulls with safe defaults
df_clean = df_clean.na.fill({"Description": "Unknown", "UnitPrice": 0.0})

# Remove duplicates by business key
df_clean = df_clean.dropDuplicates(subset=["InvoiceNo", "StockCode", "InvoiceDate"])

# Filter out zero/negative Quantity and cancellation records
df_clean = df_clean.filter(F.col("Quantity") > 0)
df_clean = df_clean.filter(~F.col("InvoiceNo").startswith("C"))

print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {df_clean.count():,}")

![108,361 rows before cleaning, 80,403 after](./screenshots/ch2_ex2_rows_before_after_cleaning.png)

*108,361 rows before cleaning, 80,403 after*

**Result:** **80,403** clean rows (~74% of the original). The order matters: rows without the key column are dropped before filling, so `na.fill()` never invents a customer; duplicates are removed on the **business key** (invoice + product + timestamp) rather than the full row; and `~` negates the cancellation filter.

**Exercise — choosing the right quality metric.** If someone accidentally comments out `na.drop(subset=["CustomerID"])`, every other step still runs without errors. The check that catches it immediately is the **null rate of `CustomerID`**: it should be 0.00%, and any value above zero means that step didn't run. Row counts alone wouldn't reveal it clearly, because they vary between uploads.

### 2.3 — Aggregating and joining efficiently

`groupBy()` partitions rows by key and `agg()` applies functions to each group in parallel. Both are **lazy** — an action triggers the work — and putting several metrics in **one `agg()` call means one scan** of the data.

In [ ]:
category_revenue = (
    df_valid
    .groupBy("Category")
    .agg(
        F.round(F.sum("Transaction_Amount"), 2).alias("total_revenue"),
        F.count("Transaction_Amount").alias("transaction_count"),
        F.round(F.avg("Transaction_Amount"), 2).alias("avg_transaction"),
    )
    .orderBy(F.col("total_revenue").desc())
)

**The hidden cost of joins — the shuffle.** In a standard join, rows with matching keys must end up on the same machine, so Spark ships rows across the network. At scale, this **shuffle** is the bottleneck. It shows up as shuffle read/write in the Spark UI (Jobs and Stages) and as exchange nodes in `.explain(mode="formatted")`.

**The fix — broadcast join.** When one side is small (a lookup or dimension table), `F.broadcast()` sends a full copy of it to every executor. The large table then never moves.

In [ ]:
df_joined = df_valid.join(df_dim, on="Category", how="left")                    # may shuffle both sides
df_broadcast = df_valid.join(F.broadcast(df_dim), on="Category", how="left")   # only the small table is copied
df_broadcast.explain(mode="formatted")

**Exercise — joining and aggregating retail data.** A new `country_lookup.csv` maps each country to its region and currency. I enriched the transactions with a broadcast join and aggregated revenue by region.

**Step 1 — load both datasets**

In [ ]:
from pyspark.sql import functions as F

# Replace with your own catalog path
path = "/Volumes/databricks_ws_***/default/datasets/online_retail.csv"
lookup_path = "/Volumes/databricks_ws_***/default/datasets/country_lookup.csv"

df = spark.read.csv(path, header=True, inferSchema=True)
df_lookup = spark.read.csv(lookup_path, header=True, inferSchema=True)

print(f"df rows        : {df.count():,}")
print(f"df_lookup rows : {df_lookup.count():,}")

**Result:** 108,361 transactions vs **44** lookup rows — a textbook case for broadcasting the small side.

**Step 2 — broadcast join on `Country`**

In [ ]:
# Add Revenue column
df_revenue = df.withColumn(
    "Revenue",
    F.col("Quantity") * F.col("UnitPrice")
)

# Broadcast join the small lookup table on Country
df_joined = df_revenue.join(
    F.broadcast(df_lookup),
    on="Country",
    how="left"
)

print(f"Joined rows: {df_joined.count():,}")
df_joined.select(
    "InvoiceNo", "Country", "Region", "Currency", "Quantity", "UnitPrice", "Revenue"
).show(5, truncate=False)

![Each transaction enriched with Region and Currency; still 108,361 rows](./screenshots/ch2_ex4_broadcast_join_preview.png)

*Each transaction enriched with Region and Currency; still 108,361 rows*

**Interpretation:** the row count is unchanged (108,361), confirming the left join didn't duplicate or lose transactions. Values like `53.199999999999996` are floating-point artifacts of `Quantity * UnitPrice`; rounding (as done later with `F.round`) is needed before reporting.

**Step 3 — verify the query plan**

In [ ]:
# Check that Spark used a broadcast join instead of a shuffle
df_joined.explain(mode="formatted")

![PhotonBroadcastHashJoin LeftOuter: only country_lookup goes through the exchange](./screenshots/ch2_ex4_broadcast_plan.png)

*PhotonBroadcastHashJoin LeftOuter: only country_lookup goes through the exchange*

**Interpretation:** the join node is **`PhotonBroadcastHashJoin LeftOuter`**. The large `online_retail` scan feeds the join directly, and the only exchange (`PhotonShuffleExchangeSink/Source`) sits above the small `country_lookup` scan — that is the broadcast copy being distributed, not a shuffle of the big table.

**Step 4 — revenue by region**

In [ ]:
# Aggregate total revenue and order count per region
region_agg = (
    df_joined
    .groupBy("Region")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("total_revenue"),
        F.count("Revenue").alias("order_count"),
    )
    .orderBy(F.col("total_revenue").desc())
)
region_agg.show(truncate=False)

![Total revenue and order count by region](./screenshots/ch2_ex4_revenue_by_region.png)

*Total revenue and order count by region*

**Interpretation:** **Europe** dominates with about **£11.35M** (shown in scientific notation, `1.134711299E7`) and 103,417 orders — far ahead of Asia (~193K). Two rows deserve follow-up: **`NULL`** (603 orders whose country isn't in the lookup, kept because of the left join) and **`Unknown`** (221 orders). In a production pipeline, unmatched keys like these should be reported as a data-quality metric.

**Exercise — understanding the shuffle bottleneck.** A standard left join of `online_retail` (100K+ rows) with `country_lookup` (44 rows) was slow, and `.explain()` showed `ShuffleExchange` nodes. "The shuffle" means Spark is **moving rows across the network between machines** so that rows with matching join keys end up together.

**Exercise — when to use a broadcast join.** For a 50-million-row `sales_transactions` table joined with a 2,000-row `product_categories` table, a broadcast join helps because the small table is **copied to every executor**, so each one can join its own partition of the large table locally — the 50 million rows never cross the network.

## Chapter 3 — Analytics and Production Pipelines

### 3.1 — Window functions: running totals and rankings

`groupBy()` returns one row per group; a **window function** adds a column and **keeps every row**, which is what running totals, rankings and row-to-row comparisons need.

In [ ]:
from pyspark.sql.window import Window

# Running total per customer over time
window_spec = (
    Window.partitionBy("Customer_ID")
    .orderBy("Date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
df_running = df_valid.withColumn(
    "running_total",
    F.round(F.sum("Transaction_Amount").over(window_spec), 2)
)

# Global ranking by total revenue
customer_totals = (
    df_valid.groupBy("Customer_ID")
    .agg(F.round(F.sum("Transaction_Amount"), 2).alias("total_revenue"))
)
rank_window = Window.orderBy(F.col("total_revenue").desc())
df_ranked = customer_totals.withColumn("revenue_rank", F.rank().over(rank_window))

`rowsBetween(Window.unboundedPreceding, Window.currentRow)` makes the frame explicit: from the customer's first transaction up to the current row.

**Exercise — ranking customers with window functions.** Which customers drive the most revenue?

**Step 1 — load the data**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Replace with your own path
path = "/Volumes/databricks_ws_***/default/datasets/online_retail.csv"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)

print(f"Rows loaded: {df.count():,}")

**Result:** `Rows loaded: 108,361`.

**Step 2 — revenue per customer**

In [ ]:
# Filter valid records
df_clean = (
    df.filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Quantity") > 0)
    .filter(~F.col("InvoiceNo").startswith("C"))
)

# Add Revenue and aggregate total per customer
customer_totals = (
    df_clean.withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))
    .groupBy("CustomerID")
    .agg(F.round(F.sum("Revenue"), 2).alias("total_revenue"))
)

print(f"Customers: {customer_totals.count():,}")
customer_totals.show(5, truncate=False)

![4,338 customers with their total revenue](./screenshots/ch3_ex1_customer_totals.png)

*4,338 customers with their total revenue*

**Step 3 — rank customers by total revenue**

In [ ]:
# Rank customers by total revenue
rank_window = Window.orderBy(F.col("total_revenue").desc())

test = customer_totals.withColumn(
    "revenue_rank",
    F.rank().over(rank_window)
)

test.filter(F.col("revenue_rank") <= 10).orderBy("revenue_rank").show(truncate=False)

![Top 10 customers by revenue](./screenshots/ch3_ex1_top10_customers.png)

*Top 10 customers by revenue*

**Result:** customer **15516** leads with **£6,544.66**, ahead of 13239 (£5,365.10) and 13737 (£5,105.94). `rank()` gives tied customers the same rank (and skips the next number); `row_number()` would break ties arbitrarily.

A window with `orderBy` and **no `partitionBy`** moves all rows to a single partition to rank them globally. That's fine for 4,338 aggregated customers, but it would be a bottleneck on millions of raw rows — which is why I aggregated first, then ranked.

### 3.2 — Structured Streaming with checkpoints

**Batch** processes a fixed dataset at once; **streaming** processes data incrementally as it arrives — ideal for transaction feeds, logs and events. With a **file-based** stream, a directory is the source: each new file becomes a micro-batch and is picked up automatically. The schema **must** be defined explicitly.

A **checkpoint** is a metadata directory recording which files have been processed. On restart, Spark resumes where it left off, so nothing is processed twice. `trigger(availableNow=True)` processes everything currently available and then stops — streaming semantics with batch-style scheduling.

**Exercise — streaming retail data into Delta Lake.** Instead of reloading everything each morning, I built a stream that picks up new files, writes them to Delta and survives restarts.

**Step 1 — prepare the stream files**

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql import functions as F

# Replace with your own path
path = "/Volumes/databricks_ws_***/default/datasets/online_retail.csv"
STREAM_DIR     = "/Volumes/databricks_ws_***/default/datasets/stream_files/"
CHECKPOINT_DIR = "/Volumes/databricks_ws_***/default/datasets/checkpoint/"
DELTA_PATH     = "/Volumes/databricks_ws_***/default/datasets/stream_output/"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)

# Split into 5 CSV files to simulate daily data drops
(
    df.limit(5000)
    .repartition(5)
    .write.mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(STREAM_DIR)
)

files = [f for f in dbutils.fs.ls(STREAM_DIR) if f.name.endswith('.csv')]
print(f"Stream files: {len(files)}")
for f in files:
    print(f"  {f.name}  ({f.size:,} bytes)")

![Five CSV part files of ~91 KB each, simulating five daily drops](./screenshots/ch3_ex2_stream_files.png)

*Five CSV part files of ~91 KB each, simulating five daily drops*

`repartition(5)` before writing produces exactly five files — one per partition.

**Step 2 — define the schema and read the stream**

In [ ]:
streaming_schema = StructType([
    StructField("InvoiceNo",   StringType(),    True),
    StructField("StockCode",   StringType(),    True),
    StructField("Description", StringType(),    True),
    StructField("Quantity",    IntegerType(),   True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice",   DoubleType(),    True),
    StructField("CustomerID",  DoubleType(),    True),
    StructField("Country",     StringType(),    True),
])

# Read the stream while passing the schema
df_stream = (
    spark.readStream.format("csv")
    .option("header", "true")
    .schema(streaming_schema)
    .load(STREAM_DIR)
)

print(f"Streaming: {df_stream.isStreaming}")

**Result:** `Streaming: True`.

**Step 3 — write to Delta with a checkpoint**

In [ ]:
# Write to Delta with checkpoint
query = (
    df_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .option("path", DELTA_PATH)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

result = spark.read.format("delta").load(DELTA_PATH)
print(f"Rows written: {result.count():,}")

progress = query.lastProgress
print(f"Rows/sec:     {progress['processedRowsPerSecond']:.0f}")

![5,000 rows written to Delta at ~1,111 rows/sec](./screenshots/ch3_ex2_rows_written.png)

*5,000 rows written to Delta at ~1,111 rows/sec*

**Step 4 — checkpoint recovery**

In [ ]:
# Restart the stream with the same checkpoint directory
query_restart = (
    df_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .option("path", DELTA_PATH)
    .trigger(availableNow=True)
    .start()
)
query_restart.awaitTermination()

progress_r = query_restart.lastProgress
rows_on_restart = progress_r["numInputRows"] if progress_r else 0
print(f"Rows on restart: {rows_on_restart}")

![The restarted stream processes 0 new rows](./screenshots/ch3_ex2_rows_on_restart.png)

*The restarted stream processes 0 new rows*

**Interpretation:** the first run wrote all **5,000** rows. The restart, using the same checkpoint, processed **0** rows, because the checkpoint already recorded those five files — no duplicates in the Delta output. `lastProgress` can be `None` when nothing was processed, hence the `if progress_r else 0` guard.

**Exercise — resuming after a restart.** After a weekend outage, the stream processed only the three new files and none from the previous week. That works because the **checkpoint directory stores metadata about which files have already been processed**.

### 3.3 — Delta Lake tables

Data kept only in memory disappears when the cluster shuts down. **Delta Lake** gives three guarantees flat files can't: **ACID transactions** (failed writes are rolled back), **schema enforcement** (mismatched types are blocked) and **versioning** (any previous state can be queried). Saving with `saveAsTable` creates a **managed table** registered in Unity Catalog.

**Exercise — writing and reading a Delta table.**

**Step 1 — load and clean**

In [ ]:
from pyspark.sql import functions as F

# Replace with your own path
path = "/Volumes/databricks_ws_***/default/datasets/online_retail.csv"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)

# Clean the data
df_clean = (
    df.filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Quantity") > 0)
    .filter(~F.col("InvoiceNo").startswith("C"))
    .withColumn("Revenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
)

print(f"Clean rows: {df_clean.count():,}")

![80,503 clean rows](./screenshots/ch3_ex4_clean_rows.png)

*80,503 clean rows*

**Interpretation:** 80,503 rows here versus 80,403 in the Chapter 2 pipeline — the difference of **100** is exactly the number of duplicates found there, because this version doesn't deduplicate.

**Step 2 — write as a managed Delta table**

In [ ]:
# Write as a managed Delta table
df_clean.write.format("delta").mode("overwrite").saveAsTable("online_retail_clean")

print(f"Rows written: {df_clean.count():,}")

![80,503 rows written to online_retail_clean](./screenshots/ch3_ex4_rows_written_delta.png)

*80,503 rows written to online_retail_clean*

**Step 3 — read back and verify**

In [ ]:
# Read the Delta table back from Unity Catalog
df_back = spark.read.table("online_retail_clean")
print(f"Rows read back: {df_back.count():,}")
df_back.show(5, truncate=False)

![The table read back by name returns the same 80,503 rows](./screenshots/ch3_ex4_read_back.png)

*The table read back by name returns the same 80,503 rows*

**Result:** the row count matches, and the table is now queryable by name from any notebook or session — unlike the temp view in Chapter 1. Delta is more reliable than CSV mainly because **ACID transactions roll back failed writes automatically**, so readers never see a half-written table.

### 3.4 — Production pipelines: Databricks Jobs and Lakeflow

A production pipeline splits the notebook into tasks with dependencies:

| Task | What it does |
|---|---|
| `task1_ingest` | loads the raw CSV, cleans it and writes a Delta table |
| `task2_revenue` | reads the cleaned table, aggregates revenue by country, writes a new table |
| `task3_top_customers` | reads the cleaned table, ranks customers by spend, saves a new table |

Jobs can be run manually, on a schedule, or by triggers.

**Exercise — building a multi-task job.** I created a job in *Jobs & Pipelines* with the three notebooks as tasks chained by dependencies, then ran it:

![First run: task1_ingest succeeded, task2_revenue failed after 2 attempts, task3 was skipped (upstream failed)](./screenshots/ch3_ex5_job_first_run_graph.png)

*First run: task1_ingest succeeded, task2_revenue failed after 2 attempts, task3 was skipped (upstream failed)*

![Timeline of the failed run, including the automatic retry of task2](./screenshots/ch3_ex5_job_first_run_timeline.png)

*Timeline of the failed run, including the automatic retry of task2*

**Debugging:** `task2_revenue` failed because it read from a table called `retail_clean`, while `task1_ingest` writes `online_retail_clean`. After fixing the table name in the notebook, I re-ran the job:

![After the fix, all three tasks succeeded](./screenshots/ch3_ex5_job_fixed_graph.png)

*After the fix, all three tasks succeeded*

![Tasks ran in sequence: ingest (36.8 s) → revenue (13.1 s) → top customers (10.7 s)](./screenshots/ch3_ex5_job_fixed_timeline.png)

*Tasks ran in sequence: ingest (36.8 s) → revenue (13.1 s) → top customers (10.7 s)*

**Interpretation:** dependencies did their job — `task3` was **skipped** instead of running on missing data. Fixing only the broken task and re-running (or using *Repair run*) is the normal workflow for multi-task jobs.

**Lakeflow (declarative pipelines).** With Jobs, I manage each step. With **Lakeflow**, I declare what each table should contain, and Databricks handles execution order, retries and compute:

In [ ]:
import dlt

@dlt.table(name="transactions_bronze")
def transactions_bronze():
    return spark.read.format("csv").schema(schema).load(FILE_PATH)

@dlt.table(name="transactions_silver")
def transactions_silver():
    return dlt.read("transactions_bronze").na.drop(...).filter(...)

@dlt.table(name="category_revenue_gold")
def category_revenue_gold():
    return dlt.read("transactions_silver").groupBy("Category").agg(...)

In the course example this pipeline ran Bronze (100K rows) → Silver (33K rows) → Gold (6 rows). The dependencies come from `dlt.read(...)`, so there's no manual task wiring.

**Exercise — why switch to Lakeflow?** The main advantage is that you **declare what each table should contain, and Databricks figures out the execution order and handles retries automatically** — no wiring tasks or managing dependencies by hand.

| Tool | Best for |
|---|---|
| Notebooks | exploring, prototyping |
| Databricks Jobs | multi-step, scheduled pipelines |
| Lakeflow | fully managed, declarative pipelines |

### 3.5 — How the Databricks pieces fit together

My summary of the platform layers used throughout the course:

| Layer | Component | Role |
|---|---|---|
| Storage | **Cloud storage** (AWS S3, Azure ADLS, Google Cloud Storage) | Databricks runs on top of the customer's storage, with external tables or managed tables whose location and lifecycle Unity Catalog manages |
| Storage format | **Delta Lake** | turns files in the lake into reliable tables with ACID transactions and versioning |
| Governance | **Unity Catalog** | centralized access control, auditing, discovery and lineage across workspaces |
| Compute | **Apache Spark and Photon** | distributed engines that run code, queries and transformations |
| Consumption | **Workspace** | where people work with the data: notebooks, Databricks SQL, machine learning and workflows |

## 🧠 Key Takeaways

- **Explicit schemas make cleaning correct**, not just tidy: inferred types can hide invalid values from `na.drop()` / `na.fill()`.
- **Clean in a deliberate order:** drop rows missing the key, fill safe defaults, deduplicate on the business key, filter invalid records — then verify with metrics (row count, duplicates, key null rate).
- **Python and SQL compile to one plan:** filters written in PySpark are pushed down into the scan, and `explain()` proves it.
- **Broadcast the small side of a join** to avoid shuffling the large table, and confirm `BroadcastHashJoin` in the plan.
- **Window functions keep every row** — running totals with explicit frames, rankings with `rank()`; aggregate first, then rank globally.
- **Checkpoints make streaming incremental and restart-safe;** `availableNow` gives streaming semantics on a batch schedule.
- **Persist to Delta, not temp views:** temp views die with the session; Delta tables are durable, transactional and governed.
- **Productionize with Jobs, evolve to Lakeflow:** task dependencies protect downstream steps; declarative pipelines remove manual orchestration.

## 💼 Application in a Data Analyst / Data Engineer Role

- **Bronze → silver ETL:** the cleaning pipeline built here (schema, nulls, dedup on business keys, invalid records, quality metrics) is a direct template for silver tables in a medallion architecture.
- **Dimension enrichment:** broadcasting lookup tables (countries, products, categories) is one of the cheapest optimizations for fact-table joins.
- **Incremental ingestion:** file-based Structured Streaming with checkpoints replaces daily full reloads, cutting compute cost and avoiding duplicates.
- **Analytics-ready outputs:** customer rankings and revenue by region are typical gold tables feeding BI dashboards.
- **Operations:** reading driver logs, debugging failed job tasks and repairing runs are everyday responsibilities once pipelines are in production.

## ✅ Skills Demonstrated

- Loading CSV files from Unity Catalog volumes into DataFrames and inspecting them with `printSchema()`, `show()` and `display()`.
- Using driver logs to troubleshoot Spark jobs.
- Shaping data with `select()` / `filter()` (combined conditions, `startswith`, `~`) and Spark SQL on temporary views.
- Reading Photon execution plans (`EXPLAIN`, `explain(mode="formatted")`) — pushed-down `DataFilters`, `ReadSchema`, broadcast vs shuffle.
- Defining `StructType` schemas and building a cleaning pipeline with `na.drop`, `na.fill`, `dropDuplicates(subset=…)` and validity filters, plus quality metrics.
- Enriching data with broadcast joins and aggregating with `groupBy().agg()`.
- Window functions: running totals with `rowsBetween` and rankings with `rank()`.
- Structured Streaming: `readStream` with an explicit schema, `writeStream` to Delta with `checkpointLocation`, `trigger(availableNow=True)`, `lastProgress` and checkpoint recovery.
- Writing and reading managed Delta tables with `saveAsTable` and `spark.read.table`.
- Building, running and debugging a multi-task Databricks Job, and explaining Lakeflow's `@dlt.table` declarative model.